# Задача ранжирования товаров по поисковому запросу


---

## 1. Постановка задачи

В рамках соревнования решается классическая задача ранжирования товаров по поисковому запросу в интернет-магазине.

**Входные данные:**
- Текстовый поисковый запрос пользователя (`query`)
- Набор товаров-кандидатов с текстовыми описаниями (`product_title`, `product_description`, `product_bullet_point`, `product_brand`, `product_color`)
- Локаль товара (`product_locale`)
- Размеченная релевантность каждого товара запросу по шкале 0-3

**Целевая переменная:**
- 3 - высоко релевантный товар
- 2 - релевантный
- 1 - частично релевантный / слабо подходящий
- 0 - нерелевантный

**Метрика качества:** nDCG@10 (Normalized Discounted Cumulative Gain на топ-10 документов)

**Seed:** 993 (фиксирован в правилах соревнования)


## 2. Анализ данных

Краткий обзор структуры данных и распределения целевой переменной.


In [1]:
import pandas as pd
import numpy as np

# Загрузка данных
train_df = pd.read_csv("../train.csv")
test_df = pd.read_csv("../test.csv")

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nКолонки: {list(train_df.columns)}")
print(f"\nРаспределение релевантности:")
print(train_df["relevance"].value_counts().sort_index())

Train shape: (49496, 11)
Test shape: (21184, 10)

Колонки: ['id', 'query_id', 'query', 'product_id', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'product_locale', 'relevance']

Распределение релевантности:
relevance
0     8027
1     3189
2    17969
3    20311
Name: count, dtype: int64


In [2]:
# Примеры данных
print("Примеры записей:")
train_df[["query", "product_title", "product_brand", "relevance"]].head(5)

Примеры записей:


,query,product_title,product_brand,relevance
0,"disturb, jeidah bila",bila sleevless enforcing bleach asymmetrical d...,unknown_brand,0
1,"#do not disturb, jeidah bila",french connection women s classic crepe light ...,French Connection,0
2,"#do disturb, jeidah bila",3d socks unisex adult animal paw crew socks - ...,Tiaronics,0
3,"#do not disturb,",bila womens sleeveless blouse 156 fenchilin to...,Bila,0
4,"#do not disturb, bila",bila womens sleeveless rayon maxi 30g unmatche...,Bila,0


In [3]:
# Статистика по пропущенным значениям
print("Пропущенные значения:")
print(train_df.isnull().sum())

Пропущенные значения:
id                          0
query_id                    0
query                       0
product_id                  0
product_title               0
product_description         1
product_bullet_point    19222
product_brand            2091
product_color           15293
product_locale              0
relevance                   0
dtype: int64


## 3. Исследованные подходы

В процессе работы были опробованы следующие подходы:

### 3.1 Базовый подход (LightGBM + MiniLM)

**Описание:**
- Базовые текстовые признаки (длина, количество слов)
- Word overlap features (Jaccard similarity)
- Семантические признаки на основе `all-MiniLM-L6-v2`
- LightGBM с RMSE loss

**Результат:** Базовый CV RMSE, но не оптимален для nDCG@10, так как RMSE не учитывает ranking

---

### 3.2 Продвинутый подход (CatBoost + Cross-Encoder)

**Улучшения:**
- Множественные embedding модели (`all-mpnet-base-v2`, `all-MiniLM-L12-v2`)
- Cross-Encoder `ms-marco-MiniLM-L-6-v2` для точной оценки релевантности пар query-product
- N-gram features (биграммы)
- Brand/color matching features
- CatBoost вместо LightGBM

**Результат:** Улучшение по сравнению с базовым подходом

---

### 3.3 Stacking подход (CatBoost + LightGBM + XGBoost)

**Описание:**
- Optuna для подбора гиперпараметров каждой модели
- LightGBM LambdaMART с ranking objective
- XGBoost с `rank:ndcg`
- Stacking с CatBoost meta-learner

**Проблемы:**
- Сложность воспроизведения (много моделей, сложный пайплайн)
- Переобучение meta-learner на OOF predictions
- Не привело к существенному улучшению относительно одиночной модели

---

### 3.4 Новый подход


**Описание:**
- SOTA embedding модели (Qwen3-Embedding)
- BGE-Reranker-v2-m3 для score-признаков
- Расширенный feature engineering
- BM25 и TF-IDF признаки
- Query-level aggregate features

**Результат:** Хорошее качество, взят за основу финального решения

---

### 3.5 Финальное решение (CatBoost PRO)

Оптимизированная версия с:
- Чистой архитектурой для инференса
- Ensemble из 6 fold-моделей
- YetiRank loss + NDCG@10 eval metric


## 4. Архитектура финального решения

### 4.1 Генерация признаков

Всего генерируется порядка 150+ признаков, разделенных на категории:

| Категория | Примерное кол-во | Описание |
|-----------|-----------------|----------|
| Basic text | ~20 | Длины текстов, количество слов, ratios |
| Word overlap | ~15 | Jaccard, word match share, exact match |
| Fuzzy matching | ~5 | Levenshtein ratio, LCS ratio |
| N-gram features | ~5 | Bigram/trigram/char-ngram overlap |
| Positional | ~10 | Позиция query в title, word counts |
| Advanced text | ~40 | Query type detection, category features, brand/color match |
| BM25 | ~5 | BM25 scores для title/desc/bullet |
| TF-IDF | ~5 | Cosine similarity |
| Semantic | ~10 | Embeddings cosine/euclidean similarity |
| Reranker | ~5 | BGE reranker scores |
| Query aggregates | ~30 | Mean/max/min/rank по query_id |


### 4.2 Примеры функций генерации признаков


In [4]:
# Примеры функций генерации признаков из features.py

from difflib import SequenceMatcher
import re


def levenshtein_ratio(s1: str, s2: str) -> float:
    """Compute Levenshtein similarity ratio (0-1)"""
    return SequenceMatcher(None, str(s1).lower(), str(s2).lower()).ratio()


def jaccard_similarity(text1: str, text2: str) -> float:
    """Compute Jaccard similarity between two texts"""
    words1 = set(str(text1).lower().split())
    words2 = set(str(text2).lower().split())
    if len(words1) == 0 or len(words2) == 0:
        return 0.0
    return len(words1 & words2) / len(words1 | words2)


def ngram_overlap(text1: str, text2: str, n: int = 2) -> float:
    """Compute n-gram overlap ratio"""

    def get_ngrams(text, n):
        words = str(text).lower().split()
        if len(words) < n:
            return set()
        return set(tuple(words[i : i + n]) for i in range(len(words) - n + 1))

    ngrams1 = get_ngrams(text1, n)
    ngrams2 = get_ngrams(text2, n)

    if not ngrams1 or not ngrams2:
        return 0.0

    intersection = ngrams1 & ngrams2
    union = ngrams1 | ngrams2
    return len(intersection) / len(union) if union else 0.0


def longest_common_substring_ratio(s1: str, s2: str) -> float:
    """Ratio of longest common substring to shorter string"""
    s1, s2 = str(s1).lower(), str(s2).lower()
    if not s1 or not s2:
        return 0.0

    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    max_len = 0

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
                max_len = max(max_len, dp[i][j])

    return max_len / min(m, n) if min(m, n) > 0 else 0.0


# Пример применения
query = "iphone case"
title = "Apple iPhone 14 Pro Case - Clear Protective Cover"

print(f"Query: {query}")
print(f"Title: {title}")
print(f"Jaccard similarity: {jaccard_similarity(query, title):.4f}")
print(f"Levenshtein ratio: {levenshtein_ratio(query, title):.4f}")
print(f"Bigram overlap: {ngram_overlap(query, title, n=2):.4f}")
print(f"LCS ratio: {longest_common_substring_ratio(query, title):.4f}")

Query: iphone case
Title: Apple iPhone 14 Pro Case - Clear Protective Cover
Jaccard similarity: 0.2222
Levenshtein ratio: 0.3667
Bigram overlap: 0.0000
LCS ratio: 0.6364


### 4.3 Семантические признаки

Для генерации семантических признаков используются:

1. **Qwen3-Embedding-0.6B** - компактная модель с хорошим качеством для retrieval задач
2. **BGE-Reranker-v2-m3** - state-of-the-art reranker для точной оценки релевантности пар

Вычисляемые признаки:
- Cosine similarity между query и title embeddings
- Cosine similarity между query и description embeddings  
- Euclidean distance между embeddings
- Reranker scores для пар (query, title) и (query, full_text)


### 4.4 Query-level Aggregate Features

Важный класс признаков - агрегаты на уровне query_id. Для каждого базового признака вычисляются:
- Среднее значение по всем кандидатам данного query
- Максимальное значение
- Минимальное значение
- Ранг текущего товара внутри query_id
- Относительное отклонение от среднего/максимума

Эти признаки позволяют модели учитывать контекст всех кандидатов для данного запроса.


## 5. Модель CatBoost

### 5.1 Конфигурация

Финальная модель обучалась с параметрами, подобранными через Optuna:

```python
params = {
    'loss_function': 'YetiRank',      # Ranking loss, оптимизирует pairwise ranking
    'eval_metric': 'NDCG:top=10',     # Целевая метрика соревнования
    'iterations': 3000-5000,
    'learning_rate': 0.02-0.05,
    'depth': 8-10,
    'l2_leaf_reg': 3-5,
    'random_strength': 0.3-0.5,
    'task_type': 'GPU',
    'random_seed': 993,
}
```

Выбор **YetiRank** вместо RMSE обусловлен тем, что это ranking loss, который напрямую оптимизирует порядок элементов, что критично для nDCG метрики.

### 5.2 Кросс-валидация

Использовалась **GroupKFold** с 5-6 фолдами и группировкой по `query_id`. Это предотвращает утечку данных между фолдами - товары одного запроса всегда находятся либо в train, либо в validation.

### 5.3 Ensemble

Финальные предсказания получены усреднением моделей со всех фолдов:
```python
predictions = sum(model.predict(X) for model in fold_models) / n_folds
```


## 6. Структура решения и инференс

### 6.1 Файловая структура


In [6]:
import os

solution_structure = """
    main.py              # Точка входа для инференса
    config.py            # Конфигурация (seed=993, модели, пути)
    features.py          # Генератор признаков (150+ features)
    inference.py         # Класс InferenceEngine
    utils.py             # Вспомогательные функции
    requirements.txt     # Зависимости с версиями
    Dockerfile           # Docker-образ для воспроизведения
    models/
        catboost_pro_best.cbm      # Лучшая модель
        catboost_pro_fold_0.cbm    # Fold модели для ensemble
        catboost_pro_fold_1.cbm
        ...
    data/
        test.csv         # Тестовые данные
    results/
        submission.csv   # Результат инференса
"""

print(solution_structure)


    main.py              # Точка входа для инференса
    config.py            # Конфигурация (seed=993, модели, пути)
    features.py          # Генератор признаков (150+ features)
    inference.py         # Класс InferenceEngine
    utils.py             # Вспомогательные функции
    requirements.txt     # Зависимости с версиями
    Dockerfile           # Docker-образ для воспроизведения
    models/
        catboost_pro_best.cbm      # Лучшая модель
        catboost_pro_fold_0.cbm    # Fold модели для ensemble
        catboost_pro_fold_1.cbm
        ...
    data/
        test.csv         # Тестовые данные
    results/
        submission.csv   # Результат инференса



## 7. Что не сработало

### 7.1 Stacking с meta-learner
- Комбинация CatBoost + LightGBM LambdaMART + XGBoost Ranking
- Meta-learner обучался на OOF predictions базовых моделей
- **Причина неудачи:** Переусложнение пайплайна, переобучение meta-learner, нестабильность результатов

### 7.2 Большие embedding модели
- Тестировались BGE-large-en-v1.5, E5-large-v2, Qwen3-Embedding-4B
- **Причина:** Marginal improvement качества при значительном увеличении времени инференса и требований к памяти

### 7.3 SVD на TF-IDF
- Латентные семантические признаки через Truncated SVD
- **Причина:** Незначительное улучшение, усложнение пайплайна, требует хранения SVD transformer

### 7.4 Cross-field features
- Перекрестные признаки title-description overlap, title-bullets similarity
- **Причина:** Высокая корреляция с уже существующими признаками, не добавляют информации


## 8. Выводы

### 8.1 Ключевые факторы успеха

1. **Правильный loss function** - YetiRank/LambdaRank напрямую оптимизируют ranking метрики, что критично для nDCG@10. Использование RMSE дает субоптимальные результаты.

2. **Качественные семантические признаки** - современные embedding и reranker модели дают значительный вклад в качество, особенно для сложных запросов где lexical matching недостаточно.

3. **Query-level aggregates** - позволяют модели учитывать контекст конкурирующих товаров и выбирать лучшего кандидата.

4. **GroupKFold валидация** - корректная валидация без утечки между query обеспечивает надежную оценку качества.

5. **Баланс сложности** - одиночная хорошо настроенная модель часто лучше сложного ensemble.

### 8.2 Потенциальные улучшения

- Fine-tuning embedding модели на domain-specific данных
- Neural ranking models (ColBERT, MonoT5)
- Более глубокий анализ ошибок и domain-specific feature engineering
